In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt

In [ ]:
#Sample Dataset
sample = pd.read_csv('flight_delay_analysis_project/2018_sample.csv')
print('Shapre of whole sample', sample.shape)
print('Shape of flights cancelled', sample.loc[sample['CANCELLED'] == 1.0].shape)
print('Shape of flights diverted', sample.loc[sample['DIVERTED'] == 1.0].shape)

In [ ]:
#Airlines and Airports are abbreviated with IATA codes, need to replace with full names
#Identify unique airlines
airlines_in_data = sample['OP_CARRIER'].unique()
print('number of airlines in dataset -', airlines_in_data.shape[0])

#read in IATA airline code data, clarify seperator and column headers
iata_ref = pd.read_csv('flight_delay_analysis_project/iata_airlines.csv', sep='^', header=None, names=['IATAcode', 'ICAOcode', 'AirlineName', 'alias'])

#selects data that returns true if airline codes match
iata_airline_ref = iata_ref[iata_ref['IATAcode'].isin(airlines_in_data)]
print('number of airline codes matched with names -', iata_airline_ref.shape[0])
iata_airline_ref.head()

In [ ]:
#Identify unique airports, splitting origin and destination
origin_in_data = sample['ORIGIN'].unique()
destination_in_data = sample['DEST'].unique()
print('number of origin airports -', origin_in_data.shape[0]) 
print('number of dest airports -', destination_in_data.shape[0])

#both origin and destination have same number of airports, to potentially simplify check whether these are all the same
origins = set(origin_in_data)
dest = set(destination_in_data)
print('origin and destination are the same -', origins == dest)


In [ ]:
#origin and destination are not the same so continue with separate dataframes for each

#read in IATA airport code data
iata_ref = pd.read_csv('flight_delay_analysis_project/iata_airports.csv')

#selects data that returns true if airline codes match
iata_origin_ref = iata_ref[iata_ref['iata_code'].isin(origin_in_data)]
iata_destination_ref = iata_ref[iata_ref['iata_code'].isin(destination_in_data)]
print('number of origin codes matched with names -', iata_origin_ref.shape[0])
print('number of destination codes matched with names -', iata_destination_ref.shape[0])

In [ ]:
#355 airports in dataset but only 354 in airport ref, finding missing one
missing_airport_origin = origin_in_data[~origin_in_data.isin(iata_origin_ref['iata_code'])]
missing_airport_destination = destination_in_data[~destination_in_data.isin(iata_destination_ref['iata_code'])]
print('missing origin -',missing_airport_origin[0]) 
print('missing destination -', missing_airport_destination[0])

In [ ]:
#both missing orgin and destination are 'ISN', quick google this is 'Sloulin Field International'
# which closed in 2019, maybe the reason for its absence as our dataset is from 2018

#add missing airport
iata_origin_ref.loc[len(iata_origin_ref)] = {'name':'Sloulin Field International Airport', 'iata_code':'ISN'}
iata_destination_ref.loc[len(iata_destination_ref)] = {'name':'Sloulin Field International Airport', 'iata_code':'ISN'}

#Rename columns so they're distinct and unique
iata_origin_ref = iata_origin_ref.rename(columns={'name': 'Origin_Name', 'iata_code':'origin_iata'})
iata_destination_ref = iata_destination_ref.rename(columns={'name': 'Destination_Name', 'iata_code':'destination_iata'})

#rerun check
print('number of origin codes matched with names -', iata_origin_ref.shape[0])
print('number of destination codes matched with names -', iata_destination_ref.shape[0])

In [ ]:
#keep only iata code and name
iata_origin_ref = iata_origin_ref[['Origin_Name', 'origin_iata']]
iata_destination_ref = iata_destination_ref[['Destination_Name', 'destination_iata']]

In [ ]:
conn = sqlite3.connect(':memory:')

#create sql table from 4 dataframes, no added index row, replaces existing table to avoid error
sample.to_sql('flights', conn, index=False, if_exists='replace')
iata_airline_ref.to_sql('airlines', conn, index=False, if_exists='replace')
iata_origin_ref.to_sql('origin_airports', conn, index=False, if_exists='replace')
iata_destination_ref.to_sql('destination_airports', conn, index=False, if_exists='replace')

In [ ]:
#use Join to match codes with full names of airlines, origins and destinations

query = """
SELECT *
FROM flights
JOIN airlines ON flights.OP_CARRIER = airlines.IATAcode
JOIN origin_airports ON flights.ORIGIN = origin_airports.origin_iata
JOIN destination_airports ON flights.DEST = destination_airports.destination_iata
"""

df = pd.read_sql_query(query, conn)
df.head()

Questions to explore
1. Some airlines suffer more delays than others
Compare delays by airline

2. Cancellations and delays are unrelated/separate
Compare delays by airline with cancellations by airline, 
do some airlines choose cancellation over delay?

3. Some airports suffer more delays
Compare delay by airport

4. Larger airports have longer taxi times
relate to 3. define large airports by volume of flights
compare taxi time to airport size


In [ ]:
#Establish complete sql 'flights' table with joined tables
df.to_sql('flights', conn, index=False, if_exists='replace')

In [ ]:
# 1.
query = """
SELECT AirlineName, arr_delay
FROM flights;
"""
result_1 = pd.read_sql_query(query,conn)
result_1['ARR_DELAY'].describe()

In [ ]:
#we have a mean of 5 but a median of -6 meaning there are outliers pulling the average up
#this is enforced by the max being a 25 hour delay
#also std is high at 46.5 considering the IQR is only 22, again sign of extreme outliers
#for boxplot limit x-axis so cluster of majority of data is clear

order = result_1.groupby('AirlineName')['ARR_DELAY'].mean().sort_values().index

result_1['AirlineName'] = pd.Categorical(result_1['AirlineName'], categories=order, ordered=True)

plt.figure(figsize=(10,6))
result_1.boxplot(column='ARR_DELAY', by='AirlineName', vert=False, showmeans=True)
plt.xlabel('Arrival Delay (mins)')
plt.title('Distribution of Arrival Delay by Airline')
#clear auto gen title
plt.suptitle('')
plt.xlim(-20,22)
plt.tight_layout()
plt.savefig('q1_delay_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
#2
query = """
SELECT
    AirlineName, 
    COUNT(*) AS total_flights,
    SUM(CASE WHEN cancelled = 1.0 THEN 1 ELSE 0 END) AS cancelled_flights,
    ROUND(100.0 * SUM(CASE WHEN cancelled = 1.0 THEN 1 ELSE 0 END)/COUNT(*) ,2) AS cancellation_pct
FROM flights
GROUP BY AirlineName
ORDER BY cancellation_pct DESC;
"""
result_2 = pd.read_sql_query(query, conn)
result_2.shape

In [ ]:
#Implementing a z-score for average arrival delay and cancellation percentage
print(result_2['cancellation_pct'].mean())
print(result_2['cancellation_pct'].std())

result_1_grouped = result_1.groupby('AirlineName')['ARR_DELAY'].mean().reset_index()

#z-score is better than normalisation as there are extreme values in this data, normalisaion would 
#clump values in small range 
result_1_grouped['delay_zscore'] = (result_1_grouped['ARR_DELAY']-result_1_grouped['ARR_DELAY'].mean())/result_1_grouped['ARR_DELAY'].std()
result_2['cancellation_zscore'] = (result_2['cancellation_pct']-result_2['cancellation_pct'].mean())/result_2['cancellation_pct'].std()
combined = pd.merge(result_1_grouped, result_2, on='AirlineName')

#.describe() should result in a mean tending towards 0 and std of 1
print(combined.describe())
combined.head()

In [ ]:
#plotting barh, first create gap column to plot representing gap in delays vs cancellations
combined['gap'] = combined['delay_zscore'] - combined['cancellation_zscore']
combined = combined.sort_values(by='gap')

colors = ['crimson' if x<0 else 'darkorange' for x in combined['gap']]

plt.figure(figsize=(10,6))
plt.barh(combined['AirlineName'], combined['gap'], color=colors)
plt.axvline(x=0, color='black', linestyle='--', linewidth=1)
plt.xlabel('<-- Cancellation Dominant    |    Delay Dominant -->')
plt.title('Difference Between Average Delay and Proportion of Cancellations by Airline')
plt.xlim(-3,2.7)
plt.tight_layout()
plt.savefig('q2_delay_cancellation_barh.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#3 some airports sufffer more delays
#Determine what type of delay can be explored
df.columns.tolist()
print('percentage of rows with security delay', round(100.0*(df['SECURITY_DELAY']>0).sum()/len(df), 2),'%')
print('percentage of rows with NAS delay', round(100.0*(df['NAS_DELAY']>0).sum()/len(df), 2),'%')

In [ ]:
#Establishing a cut off point for considering an airport large enough to contain relevant data
query = """
SELECT 
    origin_name AS "Origin", 
    COUNT(*) AS "Total Flights", 
    ROUND(100.0*SUM(CASE WHEN nas_delay > 0 THEN 1 ELSE 0 END)/COUNT(*),2) AS "NAS Delay Percentage"
FROM flights
GROUP BY origin_name
ORDER BY COUNT(*) DESC;
"""
result_3 = pd.read_sql_query(query, conn)
result_3.head()

In [ ]:
#Scatter plot
plt.figure(figsize=(10,6))
plt.scatter(result_3['Total Flights'], result_3['NAS Delay Percentage'])
plt.xlabel('Total Flights')
plt.ylabel('Percentage of Flights with NAS Delay > 0')
plt.title('Funnel Plot to Determine which Airports Contain Enough Data to be Statistically Significant')
plt.xlim(0,700)
plt.tight_layout()
plt.savefig('q3_funnel.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#identified a minimum flight-count threshold of ~130 by visually inspecting where 
#NAS delay percentage stabilized across airports of increasing size (a funnel plot), 
#rather than using an arbitrary cutoff.

#check how many airpots left after cut at 130
print('airports cut', (result_3['Total Flights'] < 130).sum())
print('airports left', (result_3['Total Flights'] >= 130).sum())

#cutting to only include reliable data
result_3_cut = result_3[result_3['Total Flights'] >= 130]

#check percentage of NAS delays more than 0 before and after cut
print('percentage of NAS Delay > 0 before cut', round(100.0*((result_3['Total Flights']*result_3['NAS Delay Percentage']/100.0).sum())/result_3['Total Flights'].sum(),2),'%')
print('percentage of NAS Delay > 0 after cut', round(100.0*((result_3_cut['Total Flights']*result_3_cut['NAS Delay Percentage']/100.0).sum())/result_3_cut['Total Flights'].sum(),2),'%')

#Filtering out airports below the 130 flight threshold changed the overall NAS delay 
#rate by only 0.02 percentage points (10.34% - 10.36%), confirming the cutoff removes 
#noise without altering the underlying pattern.

In [ ]:
#Adjust sample data to only include airports that meet minimum flight count threshold 
filtered_result_3 = df[df['Origin_Name'].isin(result_3_cut['Origin'])]

#check number of airports matches
print('unique airports', filtered_result_3['Origin_Name'].nunique())

In [ ]:
#connect to sql
filtered_result_3.to_sql('q3_data', conn, index=False, if_exists='replace')

In [ ]:
#Gather Origin and average NAS delay
query = """
SELECT Origin_Name AS 'Origin', AVG(NAS_DELAY) AS 'Delay'
FROM q3_data
GROUP BY Origin_Name
ORDER BY AVG(NAS_DELAY) ASC;
"""
new_result_3 = pd.read_sql_query(query, conn)
print(new_result_3.describe())

In [ ]:
#combine top and bottom extremes of data
result_3_extremes = pd.concat([new_result_3.head(10), new_result_3.tail(10)])

colors = ['green']*10 + ['tomato']*10

plt.figure(figsize=(10,6))
plt.barh(result_3_extremes['Origin'], result_3_extremes['Delay'], color=colors)
plt.xlabel('Average NAS Delay (mins)')
plt.title('Average NAS Delay by Airport')
plt.tight_layout()
plt.savefig('q3_NAS_delay_barh.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#4 larger airports have longer taxi times
query = """
SELECT Origin_Name as 'Airport', COUNT(*) as 'Number of Flights', AVG(TAXI_OUT) as 'Average Taxi Time'
FROM flights
GROUP BY Origin_Name
"""
result_4 = pd.read_sql_query(query, conn)
print(result_4.describe())

#check number of airports is correct
print('unique airports', len(df['Origin_Name'].unique()), ', result_4 airports', len(result_4))

In [ ]:
print('Corrolation Coefficient', round(result_4['Number of Flights'].corr(result_4['Average Taxi Time']),3))


In [ ]:
#check to see if dense cluster is driving down corr
dense_cluster = result_4[result_4['Number of Flights'] < 500]
print('Corrolation Coefficient of Airports with < 500 flights', round(dense_cluster['Number of Flights'].corr(dense_cluster['Average Taxi Time']),3))

In [ ]:
#"While the overall correlation between flight volume and average taxi time is weakly positive (r=0.258), this appears driven almost entirely by a small number of very high-volume hub airports; airports with fewer than 500 flights show virtually no relationship (r=0.046). This suggests taxi time only meaningfully increases at the most congested hubs, rather than scaling steadily with airport size across the board."
#check number of larger airports fitting the trend
print('Airports with more than 500 flights', len(result_4[result_4['Number of Flights'] > 500]))


In [ ]:
small_airports = result_4[result_4['Number of Flights']<500]
large_airports = result_4[result_4['Number of Flights']>=500]

plt.figure(figsize=(10,6))
plt.scatter(small_airports['Number of Flights'], small_airports['Average Taxi Time'], color='grey', s=small_airports['Number of Flights']/50, label='Under 500 Flights')
plt.scatter(large_airports['Number of Flights'], large_airports['Average Taxi Time'], color='green', s=large_airports['Number of Flights']/50, label='Over 500 Flights')
plt.xlabel('Total Flights')
plt.ylabel('Average Taxi Time (minutes)')
plt.title('Number of Flight vs Average Taxi Time per Airport')
plt.legend()
plt.tight_layout()
plt.savefig('q4_taxi_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print(large_airports[large_airports['Number of Flights']>9000])